In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mplfinance as mpf
import datetime as dt

def generate_random_candles(num_candles=60, start_price=100.0, seed=None):
    """
    Create random OHLC data for demonstration.
    """
    if seed is not None:
        np.random.seed(seed)

    dates = pd.date_range("2023-01-01", periods=num_candles, freq="D")

    # Simple random walk for close
    returns = np.random.normal(loc=0.0, scale=0.02, size=num_candles)
    close_prices = [start_price]
    for ret in returns[1:]:
        close_prices.append(close_prices[-1] * (1 + ret))
    close_prices = np.array(close_prices)

    # Build OHLC from close
    opens = close_prices * (1 + np.random.normal(0, 0.005, size=num_candles))
    highs = np.maximum(opens, close_prices) * (1 + np.random.uniform(0.0001, 0.01, size=num_candles))
    lows  = np.minimum(opens, close_prices) * (1 - np.random.uniform(0.0001, 0.01, size=num_candles))
    volumes = np.random.randint(100, 2000, size=num_candles)

    df = pd.DataFrame({
        "open": opens,
        "high": highs,
        "low": lows,
        "close": close_prices
    }, index=dates)
    return df

In [10]:
def find_local_extrema(series, find_min=True):
    """
    Find local minima (if find_min=True) or maxima (if find_min=False) in a timeseries.
    Returns a Series with extrema values, datetime index, and no NaNs.
    """
    values = series.values
    index = series.index
    n = len(values)
    
    if n < 3:
        return pd.Series([], index=pd.DatetimeIndex([]))
    
    if find_min:
        is_extrema = (values < np.roll(values, -1)) & (values < np.roll(values, 1))
    else:
        is_extrema = (values > np.roll(values, -1)) & (values > np.roll(values, 1))
    
    is_extrema[0] = is_extrema[-1] = False
    return pd.Series(values[is_extrema], index=index[is_extrema])

In [11]:
df = generate_random_candles(num_candles=100, start_price=100, seed=42)

In [12]:
# Example usage
# Suppose 'df' is your DataFrame with trade data, indexed by 'timestamp' and containing a 'price' column
ohlc_data = generate_random_candles(num_candles=100, start_price=100, seed=42)
 
# Access the OHLC data for each interval
# ohlc_1h = ohlc_data['1H']   # 1-hour candles
# ohlc_15m = ohlc_data['15T'] # 15-minute candles
# ohlc_5m = ohlc_data['5T']   # 5-minute candles

ohlc_5m = ohlc_data.copy()
 
 
 
i = 0
# Initialize variables
last_low, last_high = np.inf, -np.inf  # Proper extreme values
highs, lows = [], []
 
# Iterate through OHLC data
for i, row in enumerate(ohlc_5m.dropna().to_dict(orient='records')):
    o, h, l, c = row['open'], row['high'], row['low'], row['close']
    color = 'g' if c > o else 'r'
    value_dict = lambda t, v, idx, col: {'type': t, 'value': v, 'index': idx, 'color': col}
 
    # Update highs
    if h > last_high:
        if not highs or last_high < highs[-1]['value']:
            highs.append(value_dict('h', h, i, color))
        else:
            highs[-1] = value_dict('h', h, i, color)
 
    # Update lows
    if l < last_low:
        if not lows or last_low > lows[-1]['value']:
            lows.append(value_dict('l', l, i, color))
        else:
            lows[-1] = value_dict('l', l, i, color)
 
    # Update last high and low
    last_high, last_low = h, l

In [13]:
import plotly.graph_objects as go
from Utilities.dfutils import show_in_window
# Prepare the OHLC data for a 1-hour interval (as an example)
ohlc_1h = ohlc_5m[['open', 'high', 'low', 'close']].dropna()
idxs = ohlc_1h.index.values
# Create the initial candlestick chart
fig = go.Figure(data=[go.Candlestick(
    x=ohlc_1h.index,
    open=ohlc_1h['open'],
    high=ohlc_1h['high'],
    low=ohlc_1h['low'],
    close=ohlc_1h['close']
)])
 
highs_x = [ohlc_1h.index[item['index']] for item in highs]
lows_x = [ohlc_1h.index[item['index']] for item in lows]
 
# Add scatter traces for highs (green) and lows (red)
fig.add_trace(go.Scatter(
    x=highs_x,
    y=[item['value'] for item in highs],
    mode='markers',
    marker=dict(symbol='cross', color='blue', size=8),
    name='Highs'
))
 
fig.add_trace(go.Scatter(
    x=lows_x,
    y=[item['value'] for item in lows],
    mode='markers',
    marker=dict(symbol='cross', color='orange', size=8),
    name='Lows'
))
 
# Customize layout
fig.update_layout(
    title="5-min OHLC Candlestick Chart with Highs and Lows",
    xaxis_title="Time",
    yaxis_title="Price"
)
 
show_in_window(fig)

In [2]:
df = generate_random_candles(num_candles=100, start_price=100, seed=42)

df = df.reset_index()

df['min'] = np.nan
df['max'] = np.nan

for idx, row in df.iterrows():
    index_placeholder, o,h,l,c, df_min, df_max = row
    if index_placeholder == dt.datetime(2023,2,25):
        print('stop')
    if idx == 0:
        last_min = [idx,l]
        last_min2 = [idx,l]
        last_max = [idx,h]
        last_max2 = [idx,h]
        df.loc[idx, 'min'] = l
        df.loc[idx, 'max'] = h
        continue

    last_max_idx_diff = idx - last_max[0]
    last_max2_idx_diff = idx - last_max2[0]
    last_min_idx_diff = idx - last_min[0]
    last_min2_idx_diff = idx - last_min2[0]
    # If the value is maximum
    if last_max[1] < h:
        # if last_max2[1] < h:
        df.loc[idx, 'max'] = h     
        if last_max_idx_diff > 1:
            # Move maxes
            last_max2 = last_max
            last_max = [idx, h]
            # Find new minimum
            # if last_max2[1] < h:
            new_min_slice = df.loc[last_max2[0]:last_max[0], 'low']
            new_min = new_min_slice.min()
            new_min_idx = new_min_slice.idxmin()
            last_min2 = last_min
            last_min = [new_min_idx, new_min]
            df.loc[idx, 'min'] = new_min
        else:
            last_max = [idx, h]

    # If the value is minimum
    if last_min[1] > l:
        # if last_min2[1] > l:
        df.loc[idx,'min'] = l
        if last_min_idx_diff > 1:
            # Move mins
            last_min2 = last_min
            last_min = [idx, l]
            # Find new minimum
            # if last_min2[1] > l:
            new_max_slice = df.loc[last_min2[0]:last_min[0], 'high']
            new_max = new_max_slice.max()
            new_max_idx = new_max_slice.idxmax()
            last_max2 = last_max
            last_max = [new_max_idx, new_max]
            df.loc[idx, 'max'] = new_max
        else:
            last_min = [idx,l]
    

df = df.set_index('index')  


stop


In [3]:
import plotly.graph_objects as go
from Utilities.dfutils import show_in_window
# Prepare the OHLC data for a 1-hour interval (as an example)
ohlc_1h = df[['open', 'high', 'low', 'close']].dropna()
idxs = ohlc_1h.index.values
# Create the initial candlestick chart
fig = go.Figure(data=[go.Candlestick(
    x=ohlc_1h.index,
    open=ohlc_1h['open'],
    high=ohlc_1h['high'],
    low=ohlc_1h['low'],
    close=ohlc_1h['close']
)])
 
highs_x = df[['max']].dropna().index
lows_x = df[['min']].dropna().index
 
# Add scatter traces for highs (green) and lows (red)
fig.add_trace(go.Scatter(
    x=highs_x,
    y=df['max'].dropna(),
    mode='markers',
    marker=dict(symbol='cross', color='blue', size=8),
    name='Highs'
))
 
fig.add_trace(go.Scatter(
    x=lows_x,
    y=df['min'].dropna(),
    mode='markers',
    marker=dict(symbol='cross', color='orange', size=8),
    name='Lows'
))
 
# Customize layout
fig.update_layout(
    title="5-min OHLC Candlestick Chart with Highs and Lows",
    xaxis_title="Time",
    yaxis_title="Price"
)
 
show_in_window(fig)

In [16]:
df.loc[dt.datetime(2023,2,2)]

open     89.669413
high     90.844446
low      88.898410
close    90.148237
min            NaN
max      90.844446
Name: 2023-02-02 00:00:00, dtype: float64